In [0]:
DROP TABLE IF EXISTS sandbox.t_online_voc_analysis.voc_wls_key_drivers;

CREATE TABLE sandbox.t_online_voc_analysis.voc_wls_key_drivers AS (

WITH exploded_data AS (
  SELECT 
    group_dim, 
    y_feature,
    group_key,
    explode(from_json(key_drivers_json, 'ARRAY<STRUCT<x_feature:STRING, direction:STRING>>')) AS driver
  FROM sandbox.z_jungryo_lee.voc_wls_dashboard_insight_detail_v61
),

aggregated_data AS (
  SELECT 
    group_dim, 
    y_feature, 
    driver.direction,
    driver.x_feature,
    COUNT(*) AS appearance_count,
    array_join(collect_set(cast(group_key AS string)), ', ') AS group_key_list
  FROM exploded_data
  WHERE driver.direction IN ('positive', 'negative')
  GROUP BY group_dim, y_feature, driver.direction, driver.x_feature
),

ranked_drivers AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY group_dim, y_feature, direction
      ORDER BY appearance_count DESC, x_feature ASC
    ) AS rank
  FROM aggregated_data
),

final_data AS (
  SELECT 
    group_dim, 
    y_feature, 
    direction,
    x_feature, 
    appearance_count,
    group_key_list,
    rank
  FROM ranked_drivers
  WHERE appearance_count >= 2
),

all_pairs AS (
  SELECT DISTINCT
    group_dim,
    y_feature
  FROM sandbox.t_online_voc_analysis.voc_wls_dashboard_insight_detail_v6
),

missing_pairs AS (
  SELECT
    a.group_dim,
    a.y_feature
  FROM all_pairs a
  LEFT ANTI JOIN (
    SELECT DISTINCT group_dim, y_feature
    FROM final_data
  ) f
  ON a.group_dim = f.group_dim
 AND a.y_feature = f.y_feature
)

SELECT
  group_dim,
  y_feature,
  direction,
  x_feature,
  appearance_count,
  group_key_list,
  rank,
  current_timestamp() AS data_created_dt
FROM final_data

UNION ALL

SELECT
  group_dim,
  y_feature,
  'neutral' AS direction,
  'no common key drivers' AS x_feature,
  0 AS appearance_count,
  '-' AS group_key_list,
  0 AS rank,
  current_timestamp() AS data_created_dt
FROM missing_pairs

ORDER BY group_dim, y_feature, direction DESC, rank
);

select * from sandbox.t_online_voc_analysis.voc_wls_key_drivers